# Healthcare KG Population Intelligence — V5
### Read-only clinical graph + versioned NHANES population observation layer

**Purpose:** Integrate the **executed V4.2 outputs** with a **read-only copy** of the existing clinical KG. This notebook does **not** modify the original graph or access GitHub. It does not recalculate NHANES estimates, and **does not** certify outstanding epidemiological issues.

**Input:** `Healthcare_KG_Population_Intelligence_V4_2_Outputs.zip` (includes `baseline_kg_COPY.json`).

**Outputs:** a new, separate `Healthcare_KG_V5_Outputs.zip` containing observation records, links, query results, validation audit, and the untouched baseline copy. No production changes.

**Clinical guardrails:** a screening proxy is not confirmed CKD, not confirmed T2D, not a Kerendia-eligible population, and not an addressable market. **Trend claims remain blocked** while serum creatinine comparability is unresolved.

In [ ]:
from pathlib import Path
import os, json, csv, io, zipfile, hashlib, re, shutil
from datetime import datetime, timezone
import pandas as pd
import networkx as nx
ROOT=Path('/content') if Path('/content').exists() else Path.cwd()
OUT=ROOT/'Healthcare_KG_V5_Outputs'
OUT.mkdir(parents=True,exist_ok=True)
print('Working directory:',ROOT)
print('Outputs:',OUT)

## 1. Upload V4.2 archive (or use existing file)

The notebook only reads from the supplied ZIP. On Colab it prompts for upload when the archive is not present.

In [ ]:
INPUT=ROOT/'Healthcare_KG_Population_Intelligence_V4_2_Outputs.zip'
if not INPUT.exists():
    try:
        from google.colab import files
        print('Upload Healthcare_KG_Population_Intelligence_V4_2_Outputs.zip')
        uploaded=files.upload()
        candidates=[k for k in uploaded if k.lower().endswith('.zip')]
        if not candidates: raise FileNotFoundError('No ZIP file uploaded')
        INPUT=ROOT/candidates[0]
        if not INPUT.exists(): INPUT.write_bytes(uploaded[candidates[0]])
    except ImportError:
        raise FileNotFoundError(f'Place the V4.2 output ZIP at {INPUT}')
print('Input:',INPUT,'bytes:',INPUT.stat().st_size)
with zipfile.ZipFile(INPUT) as z:
    names=set(z.namelist())
    required={'baseline_kg_COPY.json','v4_2_corrected_population_estimates.csv','v4_2_validation_report.json','v4_2_cdc_methodology_audit.json','v4_2_race_ethnicity_segments.csv'}
    missing=required-names
    if missing: raise ValueError(f'Archive missing required files: {sorted(missing)}')
    baseline_bytes=z.read('baseline_kg_COPY.json')
    baseline=json.loads(baseline_bytes)
    estimates=pd.read_csv(io.BytesIO(z.read('v4_2_corrected_population_estimates.csv')))
    segments=pd.read_csv(io.BytesIO(z.read('v4_2_race_ethnicity_segments.csv')))
    report=json.loads(z.read('v4_2_validation_report.json'))
    methods=json.loads(z.read('v4_2_cdc_methodology_audit.json'))
    adj=pd.read_csv(io.BytesIO(z.read('v4_2_adjusted_jackknife.csv'))) if 'v4_2_adjusted_jackknife.csv' in names else pd.DataFrame()
    rcheck=pd.read_csv(io.BytesIO(z.read('v4_2_r_crosscheck.csv'))) if 'v4_2_r_crosscheck.csv' in names else pd.DataFrame()
print('Input cycles:',estimates.cycle.unique().tolist())
print('Baseline nodes/edges:',len(baseline['nodes']),len(baseline['edges']))

## 2. Immutable baseline verification and evidence gates

SHA-256 is checked against the V4.2 validation report; no baseline node or edge is edited. V4.2 statistical cross-checks are inherited as **reported evidence**, not re-run in V5.

In [ ]:
baseline_sha=hashlib.sha256(baseline_bytes).hexdigest()
assert baseline_sha==report['baseline_sha256'], 'Baseline hash mismatch: STOP'
assert len(baseline['nodes'])==293 and len(baseline['edges'])==517, 'Unexpected baseline graph size: STOP'
assert report.get('baseline_unchanged') is True, 'V4.2 baseline gate failed'
assert report.get('independent_r_gate')=='PASS', 'V4.2 R gate not passed'
assert methods.get('trend_claims','').startswith('BLOCKED'), 'Trend gate must remain blocked'
G=nx.MultiDiGraph()
for node in baseline['nodes']:
    nid,typ,description=node[:3]
    G.add_node(str(nid),kind=str(typ),description=str(description))
for src,tgt,rel in baseline['edges']:
    G.add_edge(str(src),str(tgt),relation=str(rel))
assert G.number_of_nodes()==293 and G.number_of_edges()==517
(OUT/'baseline_kg_COPY.json').write_bytes(baseline_bytes)
print('PASS: baseline preserved, SHA-256:',baseline_sha)
print('Inherited epidemiological gate:',report.get('epidemiological_gate'))

## 3. Population observations — separate evidence layer

Every observation carries: source release, period, scenario, screening definition, weight, point estimate, uncertainty, denominator, sample counts, provenance, and caveats. A unique ID prevents collisions. **No observations are inserted into the original clinical graph.**

In [ ]:
def finite_or_none(x):
    if pd.isna(x): return None
    return float(x)
CAVEATS=list(report.get('caveats',[]))
DEF=('Adults aged 20+ meeting notebook-defined single-visit CKD laboratory screen '
     'and probable T2D proxy; not clinically confirmed CKD or T2D')
observations=[]
for row in estimates.to_dict('records'):
    cycle=str(row['cycle']); scenario=str(row['scenario'])
    oid='NHANES_'+cycle.replace('-','_')+'_'+scenario.upper()
    observations.append({
        'id':oid,'kind':'population_observation','dataset':'NHANES',
        'period':cycle,'scenario':scenario,'definition':DEF,
        'estimated_adults':finite_or_none(row['weighted_cases']),
        'count_ci_95':[finite_or_none(row['count_ci_low']),finite_or_none(row['count_ci_high'])],
        'screening_prevalence_complete_case':finite_or_none(row['prevalence']),
        'prevalence_ci_95':[finite_or_none(row['prevalence_ci_low']),finite_or_none(row['prevalence_ci_high'])],
        'weighted_denominator':finite_or_none(row['weighted_denominator']),
        'unweighted_cases':int(row['unweighted_cases']),
        'unweighted_denominator':int(row['unweighted_denominator']),
        'survey_weight':str(row['weight']), 'design_df':int(row['design_df']),
        'provenance':{'source':'CDC NHANES','derived_from':'V4.2 output archive',
                      'archive_sha256':hashlib.sha256(INPUT.read_bytes()).hexdigest(),
                      'validation_report':'v4_2_validation_report.json',
                      'methodology_audit':'v4_2_cdc_methodology_audit.json'},
        'statistical_validation':'R_CROSSCHECK_REPORTED_PASS',
        'epidemiological_validation':report.get('epidemiological_gate','UNKNOWN'),
        'trend_comparison_allowed':False,'treatment_eligibility_estimate':False,
        'caveats':CAVEATS,
    })
assert len(observations)==len(estimates)
assert len({o['id'] for o in observations})==len(observations)
(OUT/'population_observations_v5.json').write_text(json.dumps(observations,indent=2),encoding='utf-8')
pd.DataFrame([{k:v for k,v in o.items() if not isinstance(v,(dict,list))} for o in observations]).to_csv(OUT/'population_observations_v5.csv',index=False)
print('Observations created:',len(observations))
print(pd.DataFrame([{'period':o['period'],'scenario':o['scenario'],'estimated_millions':round(o['estimated_adults']/1e6,3)} for o in observations]).to_string(index=False))

## 4. Clinical entity linking with human-review status

Link population observations to **existing** disease concepts only. Link labels are derived from graph text and deliberately marked `CANDIDATE_REVIEW_REQUIRED`; this is not a clinical ontology mapping. No links to drug eligibility are created.

In [ ]:
def candidates_for(pattern, limit=15):
    rx=re.compile(pattern,re.I)
    return [{'node_id':nid,'kind':attr['kind'],'description':attr['description'][:220]}
            for nid,attr in G.nodes(data=True) if attr['kind'].lower()=='disease' and rx.search(attr['description'])][:limit]
ckd_candidates=candidates_for('chronic kidney disease|CKD')
t2d_candidates=candidates_for('type 2 diabetes|T2DM|T2D')
print('CKD candidates:',ckd_candidates[:8])
print('T2D candidates:',t2d_candidates[:8])
links=[]
for o in observations:
    for label,cands in [('CKD screening concept',ckd_candidates),('probable T2D screening concept',t2d_candidates)]:
        for c in cands:
            links.append({'observation_id':o['id'],'clinical_node_id':c['node_id'],
                          'concept_role':label,'relation':'SCREENING_PROXY_RELATED_TO',
                          'mapping_status':'CANDIDATE_REVIEW_REQUIRED',
                          'mapping_basis':'lexical search of baseline description',
                          'asserts_confirmed_diagnosis':False})
(OUT/'population_clinical_links_v5.json').write_text(json.dumps(links,indent=2),encoding='utf-8')
print('Candidate links:',len(links),'(review required)')

## 5. Deterministic query engine with claim safeguards

This is a **retrieval/query prototype**, not an LLM agent. It refuses requests for unsupported treatment eligibility, commercial market sizing, and temporal trends. The latest NHANES estimate is returned with its scenario and uncertainty; adjusted and bridged scenarios are never silently merged.

In [ ]:
OBS_BY_ID={o['id']:o for o in observations}
def answer_population_question(question, period='2021-2023', scenario='primary'):
    q=question.lower()
    if any(term in q for term in ['eligible','eligibility','addressable market','revenue','prescription','prescribing','market size']):
        return {'status':'BLOCKED','reason':'NHANES screening proxies do not establish drug eligibility, prescribing, or market size.',
                'required':'FDA-label criteria, confirmed diagnosis, treatment history, contraindications, utilization evidence'}
    if any(term in q for term in ['trend','increas','decreas','change over time','compare years','compare cycles','growth','declin']):
        return {'status':'BLOCKED','reason':'Cross-cycle trend claims are blocked pending serum creatinine laboratory comparability and survey-method review.',
                'source':'v4_2_cdc_methodology_audit.json'}
    if any(term in q for term in ['age','sex','race','ethnic']):
        return {'status':'SEGMENT_DATA_AVAILABLE','source':'v4_2_race_ethnicity_segments.csv',
                'warning':'Subgroup estimates require reliability checks and should not be interpreted as adjusted comparisons.',
                'available_columns':list(segments.columns)}
    key='NHANES_'+period.replace('-','_')+'_'+scenario.upper()
    if key not in OBS_BY_ID:
        return {'status':'NOT_AVAILABLE','requested_id':key,'available_ids':list(OBS_BY_ID)}
    o=OBS_BY_ID[key]
    return {'status':'PRELIMINARY_SCREENING_ESTIMATE','period':period,'scenario':scenario,
            'estimated_adults':round(o['estimated_adults']),
            'count_95_ci':[round(x) for x in o['count_ci_95']],
            'complete_case_prevalence_percent':round(100*o['screening_prevalence_complete_case'],2),
            'definition':o['definition'],'weight':o['survey_weight'],
            'source_observation_id':o['id'],'validation':o['epidemiological_validation'],
            'warning':'Not a confirmed diagnosis, drug-eligible population, or trend estimate.'}
examples=[
    'How many U.S. adults screen positive for CKD and probable T2D?',
    'How many patients are eligible for Kerendia?',
    'Did CKD and T2D increase over time?',
    'How does the screening population vary by race?'
]
query_results=[{'question':q,'response':answer_population_question(q)} for q in examples]
for x in query_results: print('\nQUESTION:',x['question'],'\n',json.dumps(x['response'],indent=2))
(OUT/'example_queries_v5.json').write_text(json.dumps(query_results,indent=2),encoding='utf-8')

## 6. Export source evidence, reliability audit, and packaged results

The ZIP contains a read-only copy of the baseline KG and separately versioned V5 outputs. This notebook **never** writes to GitHub, Streamlit, or the original clinical graph.

In [ ]:
with zipfile.ZipFile(INPUT) as z:
    for fname in sorted(z.namelist()):
        if fname.startswith('v4_2_') and fname.endswith(('.csv','.json','.R')):
            (OUT/fname).write_bytes(z.read(fname))
checks={
 'version':'V5','generated_utc':datetime.now(timezone.utc).isoformat(),
 'baseline_sha256':baseline_sha,'baseline_nodes':G.number_of_nodes(),
 'baseline_edges':G.number_of_edges(),'baseline_byte_identical':(OUT/'baseline_kg_COPY.json').read_bytes()==baseline_bytes,
 'observations_count':len(observations),'candidate_links_count':len(links),
 'candidate_links_require_review':all(x['mapping_status']=='CANDIDATE_REVIEW_REQUIRED' for x in links),
 'trend_claims_blocked':answer_population_question('Did prevalence increase?')['status']=='BLOCKED',
 'eligibility_claims_blocked':answer_population_question('How many are eligible for Kerendia?')['status']=='BLOCKED',
 'source_v4_2_statistical_gate':report.get('independent_r_gate'),
 'source_v4_2_epidemiological_gate':report.get('epidemiological_gate'),
 'unresolved':['serum creatinine cross-cycle laboratory comparability','clinical concept link review',
               'screening proxy vs confirmed CKD/T2D','missing-data sensitivity assumptions',
               'external clinical/epidemiological validation'],
 'github_accessed':False,'baseline_modified':False,
 'status':'INTEGRATION_PROTOTYPE_REVIEW_REQUIRED'
}
assert checks['baseline_byte_identical'] and checks['trend_claims_blocked'] and checks['eligibility_claims_blocked']
(OUT/'v5_validation_report.json').write_text(json.dumps(checks,indent=2),encoding='utf-8')
README="""Healthcare KG V5 — read-only population intelligence integration\n\nBaseline KG is unchanged; candidate lexical links require human review.\nPopulation observations inherit V4.2 methods and REVIEW_REQUIRED epidemiological gate.\nNo eligibility, market sizing, or cross-cycle trend claims permitted.\nRun notebook cells sequentially and inspect v5_validation_report.json.\n"""
(OUT/'README.txt').write_text(README,encoding='utf-8')
OUTPUT_ZIP=ROOT/'Healthcare_KG_V5_Outputs.zip'
with zipfile.ZipFile(OUTPUT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file(): z.write(f,arcname=f.name)
print('READY:',OUTPUT_ZIP,'size:',OUTPUT_ZIP.stat().st_size)
print(json.dumps(checks,indent=2))
try:
    from google.colab import files
    files.download(str(OUTPUT_ZIP))
except ImportError:
    pass

## Next steps after V5 execution

1. Upload `Healthcare_KG_V5_Outputs.zip` for review.
2. Review and approve lexical clinical concept links (prefer ontology identifiers where available).
3. Verify unresolved laboratory comparability before enabling trend queries.
4. In V5.1, test question routing, small-sample reliability, provenance, and KG concept mapping before adding an LLM-based agent in V6.

**V5 is an integration prototype, not a clinically validated decision-support tool.**